In [1]:
import torch
import json
import random
import re
import ast
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [3]:
# load the sentences and correct labels
all_sentences = []

with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

for sentence in data:
    text = sentence["data"]["text"]
    labels = []
    results = sentence["annotations"][0]["result"]
    labels = [r["value"]["text"] for r in results]
    
    all_sentences.append({
        "text": text,
        "labels": labels
    })

In [5]:
# load the model
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [19]:
# define chat template for normal inference
def compile_ner_prompt(few_shot_examples, test_sentence):
    chat = [
          {"role": "system",
           "content": (
                "You are a helpful assistant that extracts social group mentions from text.\n"
                "Definition of a social group: A social group is a segment of society or a collection of people who share common socio-demographic traits or attributes that are ascriptive and/or acquired. \n"
                "These include characteristics like sex and gender, age, ethnicity, language, religion, place of residence, nationality, income, occupation, education and more. \n"
                "Implicit social group references such as people, everyone, communities, the public, or the nation are excluded. \n"
                "This definition excludes institutionally organized groups and state authorities like interest groups, trade unions, the police, and business entities. \n"
                "Groupings of individuals within institutionally organized groups and state authorities are included as long as the defining feature of the group is a common socio-demographic trait or attribute (e.g. workers, union members, police officers, business owners, teachers). \n"
                "Groupings based on shared beliefs, life experiences, ideology, party affiliation and/or political opinion are excluded.\n\n"
                "Your task is to extract all social group mentions from a given sentence.\n"
                "Collect all social group mentions into a single list. If there are several group mentions, this list will have several entries.\n"
                "If there are no social group mentions in the sentence, respond with: None."
                )
        }
    ]
    # add few-shot examples
    for example in few_shot_examples:
        context = example["text"]
        if example["labels"]:
            answer = str(example["labels"])
        else:
            answer = "None"
        chat.append(
            {"role": "user", "content": f"Sentence: {context}"})
        chat.append({"role": "assistant", "content": answer})
    
    # add the test sentence
    chat.append(
        {"role": "user", "content": f"Sentence: {test_sentence}"})

    # compile the prompt
    prompt = tokenizer.apply_chat_template(
    chat, return_tensors="pt", tokenize=False, add_generation_prompt=True)
    return prompt

In [7]:
# create some few-shot examples
non_empty_examples = [ex for ex in all_sentences if ex["labels"]]
empty_examples = [ex for ex in all_sentences if not ex["labels"]]
few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

# create test dataset
split_idx = int(len(non_empty_examples)*0.75)
test_dataset = non_empty_examples[split_idx:] + random.sample(empty_examples, int(len(empty_examples)*0.2))
random.shuffle(test_dataset)

In [11]:
def to_list_or_empty(entry):
    try:
        val = ast.literal_eval(entry)
        if isinstance(val, list):
            return val
        else:
            return []
    except (ValueError, SyntaxError):
        return []

In [10]:
compile_ner_prompt(few_shot_examples, "Hey there!")

'<|im_start|>system\nYou are a helpful assistant that extracts social group mentions from text.\nDefinition of a social group: A social group is a segment of society or a collection of people who share common socio-demographic traits or attributes that are ascriptive and/or acquired. \nThese include characteristics like sex and gender, age, ethnicity, language, religion, place of residence, nationality, income, occupation, education and more. \nImplicit social group references such as people, everyone, communities, the public, or the nation are excluded. \nThis definition excludes institutionally organized groups and state authorities like interest groups, trade unions, the police, and business entities. \nGroupings of individuals within institutionally organized groups and state authorities are included as long as the defining feature of the group is a common socio-demographic trait or attribute (e.g. workers, union members, police officers, business owners, teachers). \nGroupings bas

In [12]:
# generate the answers for the normal format and store in a list
gen_answers = []
for i in range(len(test_dataset)):
    sentence = test_dataset[i]["text"]
    prompt = compile_ner_prompt(few_shot_examples, sentence)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**prompt_ids)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)
    answer = generated_text.split("assistant\n")[-1]
    answer_list = to_list_or_empty(answer)
    gen_answers.append(answer_list)

In [16]:
# evaluate the generated answers
results = []

for idx in range(len(test_dataset)):
    ground_truth = test_dataset[idx]["labels"]
    prediction = gen_answers[idx]
    results.append({"labels": ground_truth,
                    "prediction": prediction})

def evaluate_predictions(results):
    y_true = []
    y_pred = []

    for example in results:
        gold_mentions = set([m.lower().strip() for m in example["labels"]])
        pred_mentions = set([m.lower().strip() for m in example["prediction"]])

        for mention in gold_mentions:
            y_true.append(1)
            y_pred.append(1 if mention in pred_mentions else 0)

        for mention in pred_mentions:
            if mention not in gold_mentions:
                y_true.append(0)
                y_pred.append(1)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return precision, recall, f1

p, r, f1 = evaluate_predictions(results)

print(f"Precision: {p:.4f} \nRecall: {r:.4f} \nF1: {f1:.4f}")

Precision: 0.1364 
Recall: 0.1667 
F1: 0.1500


In [17]:
# print some examples
for idx in range(5):
    print(test_dataset[idx]["text"])
    print(results[idx]["labels"])
    print(results[idx]["prediction"])
    print("-"*80)

"Such measures could include raising awareness of examples where local areas are taking a more informal approach to issues through, for example, restorative justice or working with potential offenders."
['potential offenders']
['restorative justice']
--------------------------------------------------------------------------------
It will protect smart meter services for both consumers and businesses by providing the enabling framework for a special administration regime for the national data and communications provider.
['consumers']
['smart meter services']
--------------------------------------------------------------------------------
I understand that there are currently 150 apprentices working on the site.
['apprentices']
['apprentices']
--------------------------------------------------------------------------------
"Friend the Member for Calder Valley (Craig Whittaker) referred-I visited him in Mytholmroyd to see some of the progress on them-and which were published last year, i